# needs to be rerun to skip created e.g.:

── adductor_magnus  (GT=3, hirr=[6]) ──
  Saved 69 rows → results_sheffield\df_adductor_magnus_hirriririir_sheffield.csv

── biceps_femoris_short  (GT=4, hirr=[11]) ──
  Saved 69 rows → results_sheffield\df_biceps_femoris_short_hirriririir_sheffield.csv

── biceps_femoris_long  (GT=5, hirr=[8]) ──
  Saved 69 rows → results_sheffield\df_biceps_femoris_long_hirriririir_sheffield.csv

── gracilis  (GT=16, hirr=[7]) ──
  Saved 69 rows → results_sheffield\df_gracilis_hirriririir_sheffield.csv

── rectus_femoris  (GT=27, hirr=[2]) ──
  Saved 69 rows → results_sheffield\df_rectus_femoris_hirriririir_sheffield.csv


── sartorius  (GT=28, hirr=[1]) ──
  Saved 69 rows → results_sheffield\df_sartorius_hirriririir_sheffield.csv

# Hirriririir (SegResNetDS) — Sheffield Dataset Evaluation

Computes per-muscle Dice / Hausdorff metrics for Hirriririir SegResNetDS segmentations on the **Sheffield** dataset.

- **Predictions**: `../sheffield_segs/Aug_N_thigh_seg.nii.gz` (labels 1–11, **bilateral** — no L/R split)
- **Ground truth**: `../../sheffeld/20440203/Aug_N_segmentations.dcm` (labels 1–37, bilateral)

Both GT and predictions are bilateral, so no OR-combining needed.

In [1]:
import glob, os, re
import numpy as np
import pandas as pd
import SimpleITK as sitk
import pydicom
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [4]:
!pwd

/c/Projects/dissector/eval_notebooks/multimodal-multiethnic/codes


In [5]:
BOUNDARY_DISTANCE = 1
GT_DIR     = os.path.join('..', '..', 'sheffeld', '20440203')
SEG_DIR    = os.path.join('..', 'sheffield_segs_water')
RESULT_DIR = 'results_sheffield'
ALGO_TAG   = 'hirriririir'
os.makedirs(RESULT_DIR, exist_ok=True)

# (muscle_name, sheffield_gt_label, hirr_labels)
# Hirriririir label map: 1=Sartorius, 2=Rectus_Femoris, 3=Vastus_Lateralis,
# 4=Vastus_Intermedius, 5=Vastus_Medialis, 6=Adductor_Magnus, 7=Gracilis,
# 8=Biceps_Femoris_Long, 9=Semitendinosus, 10=Semimembranosus, 11=Biceps_Femoris_Short
MUSCLES = [
    ('adductor_magnus',    3,  [6]),
    ('biceps_femoris_short', 4, [11]),
    ('biceps_femoris_long',  5, [8]),
    ('gracilis',           16, [7]),
    ('rectus_femoris',     27, [2]),
    ('sartorius',          28, [1]),
    ('semimembranosus',    29, [10]),
    ('semitendinosus',     30, [9]),
    ('vastus_intermedius', 35, [4]),
    ('vastus_lateralis',   36, [3]),
    ('vastus_medialis',    37, [5]),
]

def read_gt(idx):
    ds  = pydicom.dcmread(os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm'))
    raw = ds.pixel_array.astype(np.float32)
    if raw.ndim == 2: raw = raw[np.newaxis]
    labeled = np.round(raw * 37.0 / 255.0).astype(np.int32)
    labeled[raw == 0] = 0
    return np.clip(labeled, 0, 37)

def get_spacing(idx):
    ds = pydicom.dcmread(os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm'))
    ps = getattr(ds, 'PixelSpacing', [1.0, 1.0])
    st = float(getattr(ds, 'SliceThickness', 1.0))
    return (float(ps[1]), float(ps[0]), st)

seg_files = sorted(glob.glob(os.path.join(SEG_DIR, 'Aug_*_thigh_seg.nii.gz')),
                   key=lambda p: int(re.search(r'Aug_(\d+)', p).group(1)))
print(f'Found {len(seg_files)} NIfTI files')

Found 69 NIfTI files


In [ ]:
def evaluate_muscle(muscle_name, sheffield_label, hirr_labels):
    results = []
    for seg_path in seg_files:
        idx = re.search(r'Aug_(\d+)', seg_path.replace('\\', '/')).group(1)
        gt_path = os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm')
        if not os.path.exists(gt_path):
            print(f'  [skip] GT missing: Aug_{idx}'); continue
        gt_arr  = read_gt(idx)
        spacing = get_spacing(idx)
        gt_bin  = (gt_arr == sheffield_label).astype(np.uint8)

        pred_sitk = sitk.ReadImage(seg_path)
        seg_raw   = sitk.GetArrayFromImage(pred_sitk)
        if seg_raw.shape != gt_arr.shape:
            ref = sitk.GetImageFromArray(gt_arr.astype(np.int32)); ref.SetSpacing(spacing)
            pred_sitk = sitk.Resample(pred_sitk, ref, sitk.Transform(), sitk.sitkNearestNeighbor, 0)
            seg_raw   = sitk.GetArrayFromImage(pred_sitk)
        if hirr_labels is not None:
            pred_arr = np.zeros_like(seg_raw, dtype=np.uint8)
            for lbl in hirr_labels:
                pred_arr |= (seg_raw == lbl).astype(np.uint8)
        else:
            pred_arr = np.zeros(gt_arr.shape, dtype=np.uint8)

        gt_s = sitk.GetImageFromArray(gt_bin);   gt_s.SetSpacing(spacing)
        pr_s = sitk.GetImageFromArray(pred_arr); pr_s.SetSpacing(spacing)
        dice_f = sitk.LabelOverlapMeasuresImageFilter(); dice_f.Execute(gt_s, pr_s)
        if gt_bin.sum() > 0 and pred_arr.sum() > 0:
            hd_f = sitk.HausdorffDistanceImageFilter(); hd_f.Execute(gt_s, pr_s)
            hd = hd_f.GetHausdorffDistance()
        else:
            hd = np.nan
        results.append({
            'sample': f'Aug_{idx}',
            f'{muscle_name}_dice':                  dice_f.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_f.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_f.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_f.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_f.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_bin.astype(float), pred_arr.astype(float)),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_bin.astype(float), pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_bin.astype(float)),
        })
    df = pd.DataFrame(results)
    csv_path = os.path.join(RESULT_DIR, f'df_{muscle_name}_{ALGO_TAG}_sheffield.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows → {csv_path}')
    return df

dfs = {}
for muscle_name, sheffield_label, hirr_labels in MUSCLES:
    print(f'\n── {muscle_name}  (GT={sheffield_label}, hirr={hirr_labels}) ──')
    dfs[muscle_name] = evaluate_muscle(muscle_name, sheffield_label, hirr_labels)
print('\nDone.')


── adductor_magnus  (GT=3, hirr=[6]) ──
  Saved 69 rows → results_sheffield\df_adductor_magnus_hirriririir_sheffield.csv

── biceps_femoris_short  (GT=4, hirr=[11]) ──
  Saved 69 rows → results_sheffield\df_biceps_femoris_short_hirriririir_sheffield.csv

── biceps_femoris_long  (GT=5, hirr=[8]) ──
  Saved 69 rows → results_sheffield\df_biceps_femoris_long_hirriririir_sheffield.csv

── gracilis  (GT=16, hirr=[7]) ──
  Saved 69 rows → results_sheffield\df_gracilis_hirriririir_sheffield.csv

── rectus_femoris  (GT=27, hirr=[2]) ──
  Saved 69 rows → results_sheffield\df_rectus_femoris_hirriririir_sheffield.csv

── sartorius  (GT=28, hirr=[1]) ──
  Saved 69 rows → results_sheffield\df_sartorius_hirriririir_sheffield.csv

── semimembranosus  (GT=29, hirr=[10]) ──


In [ ]:
from IPython.display import display
summary_rows = []
for muscle_name, df in dfs.items():
    if df.empty: continue
    summary_rows.append({
        'muscle': muscle_name, 'n': len(df),
        'dice_mean': df[f'{muscle_name}_dice'].mean(), 'dice_std': df[f'{muscle_name}_dice'].std(),
        'hausdorff_mean': df[f'{muscle_name}_hausdorff'].mean(), 'hausdorff_std': df[f'{muscle_name}_hausdorff'].std(),
    })
summary = pd.DataFrame(summary_rows).set_index('muscle')
summary_path = os.path.join(RESULT_DIR, f'summary_{ALGO_TAG}_sheffield.csv')
summary.to_csv(summary_path)
print(f'Summary saved → {summary_path}\n')
display(summary.round(4))